# RNN Next Word Prediction Demo
Simple classroom demonstration using SimpleRNN and nursery rhyme lyrics.

In [1]:
!pip install tensorflow numpy -q


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("neisse/scrapped-lyrics-from-6-genres")

print("Path to dataset files:", path)

ModuleNotFoundError: No module named 'kagglehub'

In [3]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [4]:
import os

print(os.listdir(path))

['artists-data.csv', 'lyrics-data.csv']


In [5]:
df = pd.read_csv(
    os.path.join(path, "lyrics-data.csv")
)

In [6]:
df.head()

,ALink,SName,SLink,Lyric,language
0,/ivete-sangalo/,Arerê,/ivete-sangalo/arere.html,"Tudo o que eu quero nessa vida,\nToda vida, é\...",pt
1,/ivete-sangalo/,Se Eu Não Te Amasse Tanto Assim,/ivete-sangalo/se-eu-nao-te-amasse-tanto-assim...,Meu coração\nSem direção\nVoando só por voar\n...,pt
2,/ivete-sangalo/,Céu da Boca,/ivete-sangalo/chupa-toda.html,É de babaixá!\nÉ de balacubaca!\nÉ de babaixá!...,pt
3,/ivete-sangalo/,Quando A Chuva Passar,/ivete-sangalo/quando-a-chuva-passar.html,Quando a chuva passar\n\nPra quê falar\nSe voc...,pt
4,/ivete-sangalo/,Sorte Grande,/ivete-sangalo/sorte-grande.html,A minha sorte grande foi você cair do céu\nMin...,pt


In [7]:
df = df.dropna()

df = df.head(200)

text = " ".join(df["Lyric"].astype(str))

print("Total characters:", len(text))

Total characters: 171013


In [8]:
import re

text = text.lower()

text = re.sub(r'[^a-zA-Z\s]', '', text)

text = re.sub(r'\s+', ' ', text)

print(text[:500])

tudo o que eu quero nessa vida toda vida amar voc amar voc o seu amor como uma chama acesa queima de prazer de prazer eu j falei com deus que no vou te deixar vou te levar pra onde for qualquer lugar j fiz de tudo pra no te perder arer um lobby um hobby um love com voc arer um lobby um hobby um love com voc cai cai cai cai cai pra c hey hey hey tudotudo vai rolar meu corao sem direo voando s por voar sem saber onde chegar sonhando em te encontrar e as estrelas que hoje eu descobri no seu olhar a


In [9]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts([text])

total_words = 5000

print("Vocabulary Size:", total_words)

Vocabulary Size: 5000


In [10]:
words = text.split()

sequences = []

window_size = 5

for i in range(window_size, len(words)):

    seq = words[i-window_size:i+1]

    sequences.append(seq)

print("Total sequences:", len(sequences))

Total sequences: 33167


In [11]:
input_sequences = []

for seq in sequences:

    encoded = tokenizer.texts_to_sequences([" ".join(seq)])[0]

    if len(encoded) == window_size + 1:
        input_sequences.append(encoded)

input_sequences = np.array(input_sequences)

print(input_sequences.shape)

(33167, 6)


In [12]:
X = input_sequences[:, :-1]

y = input_sequences[:, -1]

y = to_categorical(
    y,
    num_classes=total_words
)

print(X.shape)
print(y.shape)

(33167, 5)
(33167, 5000)


In [13]:
model = Sequential([

    Embedding(
        input_dim=total_words,
        output_dim=64,
        input_length=window_size
    ),

    SimpleRNN(128),

    Dense(
        total_words,
        activation='softmax'
    )

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    X,
    y,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.0260 - loss: 6.8945 - val_accuracy: 0.0243 - val_loss: 6.8644
Epoch 2/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0287 - loss: 6.4102 - val_accuracy: 0.0297 - val_loss: 6.9401
Epoch 3/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0429 - loss: 6.0808 - val_accuracy: 0.0381 - val_loss: 6.9141
Epoch 4/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0692 - loss: 5.6490 - val_accuracy: 0.0416 - val_loss: 6.9205
Epoch 5/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.1024 - loss: 5.2304 - val_accuracy: 0.0412 - val_loss: 7.0416
Epoch 6/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1455 - loss: 4.8396 - val_accuracy: 0.0386 - val_loss: 7.1622
Epoch 7/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1887 - loss: 4.4709 - val_accuracy: 0.0431 - val_loss: 7.2424
Epoch 8/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2341 - loss: 4.1235 - val_accuracy: 0

In [16]:
def predict_next_word(seed_text):

    token_list = tokenizer.texts_to_sequences(
        [seed_text]
    )[0]

    token_list = token_list[-window_size:]

    token_list = pad_sequences(
        [token_list],
        maxlen=window_size
    )

    predicted = np.argmax(
        model.predict(token_list, verbose=0)
    )

    for word, index in tokenizer.word_index.items():

        if index == predicted:

            return word

    return ""

In [17]:
def generate_text(seed_text, n_words):

    for _ in range(n_words):

        next_word = predict_next_word(seed_text)

        seed_text += " " + next_word

    return seed_text

In [18]:
print(generate_text("we will we will", 5))

print(generate_text("hello from the", 5))

print(generate_text("i love you", 5))

we will we will we get the glow you
hello from the corazn to horas to love
i love you corazn to la dance to
